<a href="https://colab.research.google.com/github/Mkhalifa8996/Data-Science/blob/main/Transfer_learning_Feature_Extraction/Transfer_learning_Feature_Extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import zipfile
import random, os
from tensorflow.keras.models import Sequential
import matplotlib.image as mpimg
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Flatten
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import datetime

# Load the data

In [2]:
!wget https://storage.googleapis.com/ztm_tf_course/food_vision/10_food_classes_10_percent.zip

--2026-10-09 14:02:04--  https://storage.googleapis.com/ztm_tf_course/food_vision/10_food_classes_10_percent.zip
Resolving storage.googleapis.com (storage.googleapis.com)... 173.194.42.207, 74.125.132.207, 74.125.69.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|173.194.42.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 168546183 (161M) [application/zip]
Saving to: ‘10_food_classes_10_percent.zip’

10_food_classes_10_ 100%[===================>] 160.74M   232MB/s    in 0.7s    

2026-10-09 14:02:05 (232 MB/s) - ‘10_food_classes_10_percent.zip’ saved [168546183/168546183]



In [3]:
zip_ref = zipfile.ZipFile("/content/10_food_classes_10_percent.zip")
zip_ref.extractall()
zip_ref.close()

# Pass through all folders of our dataset to get the number of files and folders

In [4]:
for dirpath, dirnames, filenames in os.walk("/content/10_food_classes_10_percent"):
  print(f"There are {len(dirnames)} directories  and {len(filenames)} files in {dirpath}")

There are 2 directories  and 0 files in /content/10_food_classes_10_percent
There are 10 directories  and 0 files in /content/10_food_classes_10_percent/train
There are 0 directories  and 75 files in /content/10_food_classes_10_percent/train/ramen
There are 0 directories  and 75 files in /content/10_food_classes_10_percent/train/sushi
There are 0 directories  and 75 files in /content/10_food_classes_10_percent/train/ice_cream
There are 0 directories  and 75 files in /content/10_food_classes_10_percent/train/grilled_salmon
There are 0 directories  and 75 files in /content/10_food_classes_10_percent/train/pizza
There are 0 directories  and 75 files in /content/10_food_classes_10_percent/train/hamburger
There are 0 directories  and 75 files in /content/10_food_classes_10_percent/train/steak
There are 0 directories  and 75 files in /content/10_food_classes_10_percent/train/fried_rice
There are 0 directories  and 75 files in /content/10_food_classes_10_percent/train/chicken_curry
There are 

# **Creating the data loaders**
**Using ImageDataGenerator to load images into batches**

In [5]:
IMAGE_SHAPE = (224,224)
BATCH_SIZE = (32)

train_dir = "/content/10_food_classes_10_percent/train/"
test_dir = "/content/10_food_classes_10_percent/test/"

train_datagen = ImageDataGenerator( rescale=1/255.0)
test_datagen = ImageDataGenerator( rescale=1/255.0)

print("Training data")
train_data = train_datagen.flow_from_directory(
    train_dir,
    batch_size=BATCH_SIZE,
    target_size=IMAGE_SHAPE,
    class_mode="categorical",
)
print("Training data")
test_data = test_datagen.flow_from_directory(
    test_dir,
    batch_size=BATCH_SIZE,
    target_size=IMAGE_SHAPE,
    class_mode="categorical"
)

Training data
Found 750 images belonging to 10 classes.
Training data
Found 2500 images belonging to 10 classes.


# Setting up callbacks
**callbacks are extra functionality
you can add to your models to be performed during or after training**

1.   **TensorBoard callback: track experiments**
2.   **ModelCheckpoint callback: model check point**
1.   **EarlyStopping callback: stop a model from training(before it trains too long and overfits**


In [9]:
# Create TensorBoard callback
def create_tensorboard_callback(dir_name, experiment_name):
  log_dir = dir_name + "/" + experiment_name + "/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S%")
  tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir)
  print(f"Saving Tensorboard log file to: {log_dir}")
  return tensorboard_callback

#